# 05 -- Kalshi Market Ingestion

Fetch settled Kalshi MLB game-winner markets via dual-endpoint pagination.

**Requirement:** DATA-06

**Source:** Kalshi REST API (`GET /markets` + `GET /historical/markets`)

**Key columns:** date, home_team, away_team, kalshi_yes_price, kalshi_no_price, result, market_ticker

**Coverage:** Partial -- Kalshi MLB markets available from 2025-04-16 onward (known gap from season opener 2025-03-27 to 2025-04-15).

**Price caveat:** `kalshi_yes_price` is the settlement CLOSING price, not the pre-game opening price. Phase 4 benchmark requires pre-game price to avoid look-ahead bias.

In [ ]:
# If editable install is not set up, uncomment the next line:
# import sys; sys.path.insert(0, "..")

from src.data.kalshi import fetch_kalshi_markets
from src.data.cache import load_manifest

In [ ]:
# Fetch all settled Kalshi MLB game-winner markets
# This queries both live and historical endpoints with deduplication
df = fetch_kalshi_markets()
print(f"Total markets fetched: {len(df)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Summary: price distributions and result counts
print("Price distribution (kalshi_yes_price):")
print(df["kalshi_yes_price"].describe())

print(f"\nResult counts:")
result_counts = df["result"].value_counts(dropna=False)
print(result_counts)

none_count = df["result"].isna().sum()
if none_count > 0:
    print(f"\nNOTE: {none_count} markets have result=None (voided/postponed games)")
    print("These are excluded from win-probability training by design.")

print(f"\nSample markets:")
display(df[["date", "home_team", "away_team", "kalshi_yes_price", "kalshi_no_price", "result", "market_ticker"]].head(10))

In [ ]:
import pandas as pd

# Coverage validation: explicit reporting of known March 27 - April 16 gap
if len(df) > 0 and "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    min_date, max_date = df["date"].min(), df["date"].max()
    # Known gap: Kalshi MLB markets only available from 2025-04-16 onward.
    # The 2025 season opened 2025-03-27 -- ~20 days (~60-80 games) are permanently missing.
    gap_start = pd.Timestamp("2025-03-27")
    gap_end   = pd.Timestamp("2025-04-16")
    print(f"Date range in cache: {min_date.date()} to {max_date.date()}")
    print(f"Known coverage gap:  {gap_start.date()} to {(gap_end - pd.Timedelta(days=1)).date()} (~20 days, no Kalshi markets)")
    print(f"Total markets: {len(df)}")
    gap_rows = df[(df["date"] >= gap_start) & (df["date"] < gap_end)]
    if len(gap_rows) > 0:
        print(f"WARNING: {len(gap_rows)} markets found inside gap period -- unexpected, verify.")
    else:
        print("Gap period confirmed empty (expected -- Kalshi did not offer MLB markets before 2025-04-16).")
    print(f"\nNOTE for Phase 4: backtest join against Kalshi covers 2025-04-16+ only.")
    print(f"  Teams/games before this date have no Kalshi benchmark price available.")
else:
    print("No data returned or missing 'date' column. Check API connectivity and authentication.")

In [ ]:
# Team coverage: which teams appear in Kalshi markets?
if len(df) > 0:
    print("Home teams:")
    home_counts = df["home_team"].value_counts(dropna=False)
    print(home_counts)
    
    print(f"\nAway teams:")
    away_counts = df["away_team"].value_counts(dropna=False)
    print(away_counts)
    
    # How many teams could not be parsed?
    unparsed_home = df["home_team"].isna().sum()
    unparsed_away = df["away_team"].isna().sum()
    if unparsed_home > 0 or unparsed_away > 0:
        print(f"\nWARNING: {unparsed_home} rows with unparsed home_team, {unparsed_away} with unparsed away_team")
        print("Sample unparsed titles:")
        unparsed = df[df["home_team"].isna() | df["away_team"].isna()]
        display(unparsed[["market_ticker", "title", "subtitle"]].head(5))

In [ ]:
# Cache check
manifest = load_manifest()
kalshi_keys = {k: v for k, v in manifest.items() if k.startswith("kalshi_")}
print(f"Cached Kalshi files: {len(kalshi_keys)}")
for k, v in sorted(kalshi_keys.items()):
    print(f"  {k}: {v['row_count']} rows, fetched {v['fetch_date'][:10]}")